In [0]:
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import *

class DatabricksObservability:

    def __init__(self, catalog_name, environment="dev"):
        self.catalog = catalog_name
        self.environment = environment
        self.execution_id = None
        self.start_time = None
        self.spark = SparkSession.builder.getOrCreate()

    def start_execution(self, job_name):
        self.execution_id = f"{job_name}_{int(time.time())}"
        self.start_time = datetime.now()
        return self.execution_id

    def log_ingestion_metric(
        self,
        source_table,
        ingestion_type,
        start_time,
        end_time,
        records_loaded,
        landing_path,
        status="success"
    ):
        try:

            duration = (end_time - start_time).total_seconds()

            data = [(
                self.execution_id,
                "silver",
                source_table,
                ingestion_type,
                self.environment,
                start_time,
                end_time,
                duration,
                records_loaded,
                records_loaded,
                landing_path,
                status,
                datetime.now()
            )]

            columns = [
                "execution_id","source_system","source_table","ingestion_type","environment",
                "start_time","end_time","duration_seconds","records_extracted","records_loaded",
                "landing_path","status","created_at" 
            ]

            df = self.spark.createDataFrame(data, columns)
            df.write.mode("append").saveAsTable(f"{self.catalog}.observability.ingestion_metrics")

        except Exception as e:
            print(f"Erro ingestion: {e}")

    def log_job_execution(
        self,
        job_name,
        layer,
        start_time,
        end_time,
        status,
        records_processed,
        records_failed=0,
        source_system="silver",
        error_message=None,
        metadata=None
    ):
        try:

            import json

            metadata = json.dumps(metadata) if metadata else ""
            error_message = error_message or ""
            duration = (end_time - start_time).total_seconds()

            data = [(
                self.execution_id,
                job_name,
                layer,
                self.environment,
                start_time,
                end_time,
                duration,
                status,
                records_processed,
                records_failed,
                source_system,
                error_message,
                datetime.now()
            )]

            columns = [
                "execution_id","job_name","layer","environment","start_time","end_time",
                "duration_seconds","status","records_processed","records_failed",
                "source_system","error_message","created_at"
            ]

            df = self.spark.createDataFrame(data, columns)
            df.write.mode("append").saveAsTable(f"{self.catalog}.observability.job_execution_metrics")

        except Exception as e:
            print(f"Erro job_execution: {e}")

    def log_data_quality(
        self,
        table_name,
        layer,
        check_type,
        check_name,
        status,
        records_checked,
        records_passed,
        records_failed=0,
        threshold=100.0
    ):
        try:
            pass_rate = (records_passed / records_checked * 100) if records_checked > 0 else 0
            check_id = f"{self.execution_id}_dq"

            data = [(
                check_id,
                self.execution_id,
                table_name,
                layer,
                check_type,
                check_name,
                status,
                records_checked,
                records_passed,
                records_failed,
                pass_rate,
                threshold,
                datetime.now()
            )]

            columns = [
                "check_id","execution_id","table_name","layer","check_type","check_name",
                "status","records_checked","records_passed","records_failed",
                "pass_rate","threshold","created_at"
            ]

            df = self.spark.createDataFrame(data, columns)
            df.write.mode("append").saveAsTable(f"{self.catalog}.observability.data_quality_metrics")

        except Exception as e:
            print(f"Erro data_quality: {e}")

In [0]:
display(spark.sql("DESCRIBE DETAIL bikestore.gold.sales_ny").collect()[0])


In [0]:
tables = spark.sql("SHOW TABLES IN bikestore.gold")

table_names = []

for table in tables.collect():
    table_names.append(table.tableName)

print(table_names)

In [0]:
layer = "gold"

for table_name in table_names:

    full_table_name = (f"bikestore.{layer}.{table_name}")
    print(f"""
    Processing:
    full_table_name: {full_table_name}
    table_name = {table_name}
    layer = {layer}
    """)


In [0]:

files = full_table_name


print(f"layer = {layer}")

In [0]:
type(table_names)

In [0]:
%sql
select * from bikestore.gold.sales_ny

In [0]:
# table_names = ["product"]

for table_name in table_names:

    full_table_name = (f"bikestore.{layer}.{table_name}")
    print(f"""
    Processing:
    full_table_name: {full_table_name}
    table_name = {table_name}
    layer = {layer}
    """)


In [0]:
layer = "gold"

tables = spark.sql(f"SHOW TABLES IN bikestore.{layer}")

table_names = []

for table in tables.collect():
    table_names.append(table.tableName)

print(table_names)

In [0]:
from pyspark.sql.functions import current_timestamp, col, lit
from datetime import datetime

# observabilidade

layer = "gold"

obs = DatabricksObservability("bikestore", "dev")

job_start = datetime.now()

for table_name in table_names:

    full_table_name = (f"bikestore.{layer}.{table_name}")
    print(f"""
    Processing:
    full_table_name: {full_table_name}
    table_name = {table_name}
    layer = {layer}
    """)

    obs.start_execution(f"{layer}_{table_name}")

    job_start = datetime.now()

    try:
        #loading
        df = (
            spark.table(full_table_name)
            .withColumn("gold_ingest_timestamp", current_timestamp())
            .withColumn("gold_source_table", lit(full_table_name))
        )

        count = df.count()

        # data quality
        obs.log_data_quality(
            table_name=table_name,
            layer=layer,
            check_type="completeness",
            check_name="row_count_validation",
            status="success" if count > 0 else "failed",
            records_checked=count,
            records_passed=count if count > 0 else 0,
            records_failed=0 if count > 0 else count,
            threshold=100.0
    )

        df.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(
                f"{obs.catalog}.{layer}.{table_name}"
            )

        job_end = datetime.now()

        # ingestion
        obs.log_ingestion_metric(
            source_table=table_name,
            ingestion_type="batch",
            start_time=job_start,
            end_time=job_end,
            records_loaded=count,
            landing_path=full_table_name,
            status="success"
        )

        # execution
        obs.log_job_execution(
            job_name=table_name,
            layer=layer,
            start_time=job_start,
            end_time=job_end,
            status="success",
            records_processed=count
        )

        print(f"{table_name} processada")

    except Exception as e:

        job_end = datetime.now()

        obs.log_job_execution(
            job_name=table_name,
            layer=layer,
            start_time=job_start,
            end_time=job_end,
            status="failed",
            records_processed=0,
            error_message=str(e)
        )

        print(f"Erro em {table_name}: {e}")